In [ ]:
import time
import pandas as pd
from shapely.geometry import Point, box
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import geopandas as gpd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from pyproj import CRS, Transformer
import numpy as np
from shapely.wkt import loads
from shapely.geometry import Point, mapping
from shapely.ops import unary_union
from shapely.ops import linemerge
from shapely.geometry import Polygon
import os
import base64


In [ ]:
# ### data folder, output folder and models

webscrapping_path = r"E:\Base_map_pipeline\Jobs\SAVE FILE\webscrap"
os.makedirs(webscrapping_path, exist_ok=True)


shape_path = r"E:\Base_map_pipeline\Jobs\SAVE FILE\11.shp"

In [4]:
## Grid creation

polygon_gdf = gpd.read_file(shape_path) ## need to automate this 

coordinate_point = polygon_gdf.geometry.iloc[0].centroid

# Get the bounds of the buffer
poly = polygon_gdf.geometry.iloc[0]
minx, miny, maxx, maxy = poly.bounds

# Debugging - print the buffer bounds to check if they are symmetrical
print(f"poly bounds: minx={minx}, miny={miny}, maxx={maxx}, maxy={maxy}")

# Ensure grid is divided equally left and right from the center
grid_width = 0.00061
grid_height = 0.00060

# Step 3: Divide the buffer area into an n x n grid
n = int((maxx - minx) / grid_width)  # number of columns
m = int((maxy - miny) / grid_height)  # Set n for grid division, i.e., 5x5

grid_number = max(n,m) ### number of grids created
print(f"Dividing the buffer into a {grid_number}x{grid_number} grid...")
x_start1= []
x_end1 = []
y_start1 = []
y_end1 = []

# Step 4: Create grid boxes within the buffer
grid = []
for i in range(grid_number):
    for j in range(grid_number):
        x_start = minx + i * grid_width
        x_end = minx + (i + 1) * grid_width
        y_start = miny + j * grid_height
        y_end = miny + (j + 1) * grid_height
        grid.append(box(x_start, y_start, x_end, y_end))
        x_start1.append(x_start)
        x_end1.append(x_end)
        y_start1.append(y_start)
        y_end1.append(y_end)

# Step 5: Convert grid boxes into a GeoDataFrame
print("Converting grid boxes into a GeoDataFrame...")
grid_gdf = gpd.GeoDataFrame(geometry=grid)

# Step 6: Get the centroid of each grid box
print("Calculating centroids of the grid boxes...")
grid_gdf['centroid'] = grid_gdf.centroid
grid_gdf['x_start']= x_start1
grid_gdf['x_end']= x_end1
grid_gdf['y_start']= y_start1
grid_gdf['y_end']= y_end1


poly bounds: minx=-120.9475824877918, miny=37.677926404792544, maxx=-120.94087685004558, maxy=37.68512577956461
Dividing the buffer into a 11x11 grid...
Converting grid boxes into a GeoDataFrame...
Calculating centroids of the grid boxes...


In [5]:
#### code for nXn grid
def create_fixed_cell_nxn_grid(shape_path, cell_width_meters=100, cell_height_meters=100):
    """
    Create an N×N grid with strictly fixed cell dimensions in meters.
    Grid may extend beyond the shapefile bounds to maintain an N×N structure.
    
    Parameters:
    shape_path (str): Path to the shapefile
    cell_width_meters (float): Fixed width of each cell in meters
    cell_height_meters (float): Fixed height of each cell in meters
    
    Returns:
    gpd.GeoDataFrame: GeoDataFrame containing grid cells with their properties
    """
    # Read the shapefile
    polygon_gdf = gpd.read_file(shape_path)
    
    # Get the polygon from the shapefile
    poly = polygon_gdf.geometry.iloc[0]
    
    # Get the bounds of the polygon
    minx, miny, maxx, maxy = poly.bounds
    
    # Calculate the centroid for the projection
    centroid_lon = (minx + maxx) / 2
    centroid_lat = (miny + maxy) / 2
    
    # Create a custom transverse Mercator projection centered on the shapefile
    custom_crs = CRS(
        proj="tmerc",
        lat_0=centroid_lat,
        lon_0=centroid_lon,
        ellps="WGS84",
        units="m"
    )
    
    # Create transformers for projection and inverse projection
    to_projected = Transformer.from_crs(4326, custom_crs, always_xy=True)
    to_geographic = Transformer.from_crs(custom_crs, 4326, always_xy=True)
    
    # Convert polygon bounds to projected coordinates (meters)
    x0, y0 = to_projected.transform(minx, miny)
    x1, y1 = to_projected.transform(maxx, maxy)
    
    # Calculate the width and height in meters
    width_meters = x1 - x0
    height_meters = y1 - y0
    
    print(f"Shape dimensions in meters: width={width_meters:.2f}m, height={height_meters:.2f}m")
    
    # Calculate the number of cells needed in each direction
    n_cells_x = int(np.ceil(width_meters / cell_width_meters))
    n_cells_y = int(np.ceil(height_meters / cell_height_meters))
    
    # Determine N for the N×N grid (maximum of width and height cell counts)
    n = max(n_cells_x, n_cells_y)
    
    print(f"Creating an {n}×{n} grid with fixed cell dimensions of {cell_width_meters}m × {cell_height_meters}m")
    print(f"Original shape would require {n_cells_x}×{n_cells_y} cells, extending to {n}×{n} for square grid")
    
    # Create grid cells with properties
    grid_cells = []
    cell_properties = {
        'x_start_proj': [], 'x_end_proj': [],
        'y_start_proj': [], 'y_end_proj': [],
        'x_start': [], 'x_end': [],
        'y_start': [], 'y_end': []
    }
    
    for i in range(n):
        for j in range(n):
            # Calculate cell coordinates in projected system using FIXED cell dimensions
            x_start_proj = x0 + i * cell_width_meters
            x_end_proj = x0 + (i + 1) * cell_width_meters
            y_start_proj = y0 + j * cell_height_meters
            y_end_proj = y0 + (j + 1) * cell_height_meters
            
            # Convert corners to geographic coordinates
            x_start_geo, y_start_geo = to_geographic.transform(x_start_proj, y_start_proj)
            x_end_geo, y_end_geo = to_geographic.transform(x_end_proj, y_end_proj)
            
            # Additional corner coordinates needed to properly define the polygon
            x_start_end_geo, y_start_end_geo = to_geographic.transform(x_start_proj, y_end_proj)
            x_end_start_geo, y_end_start_geo = to_geographic.transform(x_end_proj, y_start_proj)
            
            # Create a polygon with the geographic coordinates
            cell_polygon = Polygon([
                [x_start_geo, y_start_geo],           # bottom-left
                [x_end_start_geo, y_end_start_geo],   # bottom-right
                [x_end_geo, y_end_geo],               # top-right
                [x_start_end_geo, y_start_end_geo],   # top-left
                [x_start_geo, y_start_geo]            # close the polygon
            ])
            
            grid_cells.append(cell_polygon)
            
            # Store cell properties
            cell_properties['x_start_proj'].append(x_start_proj)
            cell_properties['x_end_proj'].append(x_end_proj)
            cell_properties['y_start_proj'].append(y_start_proj)
            cell_properties['y_end_proj'].append(y_end_proj)
            cell_properties['x_start'].append(x_start_geo)
            cell_properties['x_end'].append(x_end_geo)
            cell_properties['y_start'].append(y_start_geo)
            cell_properties['y_end'].append(y_end_geo)
    
    # Create GeoDataFrame with the grid cells
    grid_gdf = gpd.GeoDataFrame(
        cell_properties,
        geometry=grid_cells,
        crs=4326  # WGS84 geographic coordinates
    )
    
    # Calculate centroids
    grid_gdf['centroid'] = grid_gdf.geometry.centroid
    
    # Add cell indices for easier reference
    grid_gdf['cell_row'] = [j for i in range(n) for j in range(n)]
    grid_gdf['cell_col'] = [i for i in range(n) for j in range(n)]
    grid_gdf['cell_id'] = [f"{i}_{j}" for i in range(n) for j in range(n)]
    
    # Add a flag to identify cells that intersect with the original polygon
    grid_gdf['intersects_shape'] = grid_gdf.intersects(poly)
    
    return grid_gdf, custom_crs, n


In [6]:
# Example usage
 # Replace with your shapefile path
grid_gdf, custom_crs, grid_number = create_fixed_cell_nxn_grid(
    shape_path,
    cell_width_meters=52.6608,  # Adjust cell dimensions as needed
    cell_height_meters= 66.5999990 
)

# Save the grid to a file
# grid_gdf.to_file("distance_based_grid.shp")

# Visualize (optional)
original_gdf = gpd.read_file(shape_path)
# ax = original_gdf.plot(alpha=0.5, color='red')
# grid_gdf.plot(ax=ax, alpha=0.3, edgecolor='black')

Shape dimensions in meters: width=591.51m, height=799.06m
Creating an 12×12 grid with fixed cell dimensions of 52.6608m × 66.599999m
Original shape would require 12×12 cells, extending to 12×12 for square grid


In [7]:
grid_gdf = grid_gdf[['geometry','centroid','x_start','x_end','y_start','y_end']]
grid_gdf

,geometry,centroid,x_start,x_end,y_start,y_end
0,"POLYGON ((-120.94758 37.67793, -120.94699 37.6...",POINT (-120.94728 37.67823),-120.947582,-120.946986,37.677926,37.678526
1,"POLYGON ((-120.94758 37.67853, -120.94699 37.6...",POINT (-120.94728 37.67883),-120.947583,-120.946986,37.678526,37.679127
2,"POLYGON ((-120.94758 37.67913, -120.94699 37.6...",POINT (-120.94728 37.67943),-120.947583,-120.946986,37.679127,37.679727
3,"POLYGON ((-120.94758 37.67973, -120.94699 37.6...",POINT (-120.94728 37.68003),-120.947583,-120.946986,37.679727,37.680327
4,"POLYGON ((-120.94758 37.68033, -120.94699 37.6...",POINT (-120.94728 37.68063),-120.947583,-120.946986,37.680327,37.680927
...,...,...,...,...,...,...
139,"POLYGON ((-120.94102 37.68213, -120.94042 37.6...",POINT (-120.94072 37.68243),-120.941016,-120.940419,37.682127,37.682727
140,"POLYGON ((-120.94102 37.68273, -120.94042 37.6...",POINT (-120.94072 37.68303),-120.941016,-120.940419,37.682727,37.683327
141,"POLYGON ((-120.94102 37.68333, -120.94042 37.6...",POINT (-120.94072 37.68363),-120.941016,-120.940419,37.683327,37.683927
142,"POLYGON ((-120.94102 37.68393, -120.94042 37.6...",POINT (-120.94072 37.68423),-120.941016,-120.940419,37.683927,37.684527


In [8]:
# To save grid data 
grid_gdf.to_csv(os.path.join(webscrapping_path, "Grid_lat_lon.csv"), index = False, header=True)

In [ ]:
### Sonika Code 
import re
import time
import base64
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.action_chains import ActionChains

# Setup Edge WebDriver
edge_driver_path = r"C:\Users\Downloads\edgedriver_win64_142_94\msedgedriver.exe"

edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--no-sandbox")
edge_options.add_argument("--disable-dev-shm-usage")
edge_options.page_load_strategy = 'normal'

service = Service(edge_driver_path)
driver = webdriver.Edge(service=service, options=edge_options)

# Constants
expected_a = "29.23614102"
expected_d = "152.13864295"

# Prepare data
x = grid_gdf.centroid  
url_list = []
image_list = []

for i in tqdm(range(len(x))):
    lat = x[i].y
    lon = x[i].x

    url = f"https://earth.google.com/web/@{lat},{lon},{expected_a}a,{expected_d}d,35.5y,0h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"

    reload_attempts = 0
    max_attempts = 3
    final_url = url

    while reload_attempts < max_attempts:
        driver.get(url)

        try:
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
        except TimeoutException:
            print(f"Timeout while loading: lat={lat}, lon={lon}")
            break

        ### Temporary closed as it can be done by manually here as well
        # Only try dismissing popup once
        # try:
        #     time.sleep(2)
        #     ActionChains(driver).move_by_offset(10, 10).click().perform()
        #     ActionChains(driver).move_by_offset(-10, -10).perform()  # Reset offset
        #     print("Clicked top-left to dismiss 'New from Google Earth' popup.")
        # except Exception as e:
        #     print(f"Popup click failed: {e}")
            
        #     # Then, survey popup
        # try:
        #     survey_button = WebDriverWait(driver, 15).until(
        #         EC.presence_of_element_located((By.XPATH, "//button[contains(text(), 'Take survey')]"))
        #     )
        #     close_x = survey_button.find_element(By.XPATH, "./following-sibling::button | .//following::button[@aria-label='Close'][1]")
        #     close_x.click()
        #     time.sleep(2)
        # except TimeoutException:
        #     print("No 'Help improve Google Earth' banner found.")


        time.sleep(7)
        current_url = driver.current_url

        match = re.search(r'@[^,]+,[^,]+,([^\s,]+)a,([^\s,]+)d', current_url)
        if match:
            current_a = match.group(1)
            current_d = match.group(2)

            if current_a != expected_a or current_d != expected_d:
                print(f"Mismatch at lat={lat}, lon={lon}: a={current_a}, d={current_d} — retrying ({reload_attempts+1})...")
                reload_attempts += 1
                continue
            else:
                final_url = current_url
                break
        else:
            print(f"Could not parse 'a' and 'd' from URL: {current_url}")
            break

    # Proceed with screenshot as before
    try:
        url_list.append(final_url)
        driver.get(final_url)
        time.sleep(7)

        screenshot_base64 = driver.get_screenshot_as_base64()
        screenshot_base64 = driver.get_screenshot_as_base64()
        screenshot_bytes = BytesIO(base64.b64decode(screenshot_base64))
        img = Image.open(screenshot_bytes)

        # Crop to target region
        crop_region = (577, 118, 1023, 682)  # Adjust if needed
        cropped_img = img.crop(crop_region)
        image_list.append(cropped_img)
    except Exception as e:
        print(f"Screenshot failed at lat={lat}, lon={lon}: {e}")
        
driver.quit()


100%|██████████| 144/144 [38:37<00:00, 16.09s/it]


In [14]:
# 

In [15]:
grid_gdf['image'] = image_list
grid_gdf['url'] = url_list

In [16]:
grid_gdf.to_csv(os.path.join(webscrapping_path, "Grid_lat_lon_with_url.csv"), index = False, header=True)

In [17]:
# save images to folder

import math
n = grid_number   # Example: You can adjust this value depending on your grid size ### need to automate this 

# Loop through image_list and save each image
for i in tqdm(range(0, len(image_list))):
    a = math.floor(i / n) + 1
    b = (i % n) + 1
    # Print a, b if you need to check the grid division
    # print(f"a: {a}, b : {b}")
    
    # Construct the filename using os.path.join
    filename = os.path.join(webscrapping_path, rf"grid{a,b}.png")
    
    # Ensure you are saving the image from the correct list (image_list)
    image_list[i].save(filename)

100%|██████████| 144/144 [00:07<00:00, 20.21it/s]


## Create a final image

In [18]:
import os
from PIL import Image

def stitch_grid_images_parentheses(image_folder, output_tif, tile_size, background_color=(0, 0, 0)):
    """
    Stitches a folder of images into one large image based on their grid position
    extracted from the filename in the format 'grid(row, col).png'.
    Saves the output as a TIFF file.

    Args:
        image_folder (str): Path to the folder containing the image tiles.
        output_tif (str): Path to save the stitched TIFF file.
        tile_size (tuple): A tuple (width, height) representing the size of each tile in pixels.
        background_color (tuple): RGB color to use for empty grid spaces (default is black).
    """
    images = {}
    max_row = -1
    max_col = -1

    # Regular expression to find row and column numbers in filenames like 'grid(1, 4).png'
    import re
    regex = re.compile(r"grid\((\d+),\s*(\d+)\)\.")
    #regex = re.compile(r"grid\((\d+),\s*(\d+)\)_result\.png$")


    # Read images and extract grid numbers
    for filename in os.listdir(image_folder):
        match = regex.search(filename)
        print(match)
        if match:
            try:
                row = int(match.group(1))-1
                col = int(match.group(2))-1
                print(row, col)
                img_path = os.path.join(image_folder, filename)
                img = Image.open(img_path)
                if img.size == tile_size:
                    images[(row, col)] = img
                    max_row = max(max_row, row)
                    max_col = max(max_col, col)
                else:
                    print(f"Warning: Image {filename} has incorrect size {img.size}, expected {tile_size}. Skipping.")
            except ValueError:
                print(f"Warning: Could not parse row/col from filename {filename}. Skipping.")
            except Exception as e:
                print(f"Warning: Could not open image {filename}: {e}. Skipping.")
    print(max_row, max_col)
    if not images:
        print("Error: No valid grid images found.")
        return

    # Determine the dimensions of the final stitched image
    stitched_width = (max_col + 1) * tile_size[0]
    stitched_height = (max_row + 1) * tile_size[1]
    stitched_image = Image.new("RGB", (stitched_width, stitched_height), background_color)

    # Paste the individual images onto the stitched image
    for (col, row), img in images.items():
        x_offset = col * tile_size[0]
        y_offset = (max_row - row) * tile_size[1]
        # img = img.rotate(180)
        stitched_image.paste(img, (x_offset, y_offset))

    # Save the stitched image as a TIFF file
    try:
        stitched_image.save(output_tif, "TIFF")
        print(f"Stitched image saved as {output_tif}")
    except Exception as e:
        print(f"Error saving TIFF file: {e}")

image_folder = webscrapping_path
# image_folder =r"D:\MH_FH_base_map_APN_new_jobs_pipeline\A05B9DY shape file\output"
# output_tif_file = r"D:\MH_FH_base_map_APN_new_jobs_pipeline\A05B9DY shape file\output\output.tif"
output_tif_file = os.path.join(os.path.dirname(webscrapping_path), "stitched_imag_2.tif")
tile_width = 446
tile_height = 564
expected_tile_size = (tile_width, tile_height)

stitch_grid_images_parentheses(image_folder, output_tif_file, expected_tile_size)

<re.Match object; span=(0, 11), match='grid(1, 1).'>
0 0
<re.Match object; span=(0, 12), match='grid(1, 10).'>
0 9
<re.Match object; span=(0, 12), match='grid(1, 11).'>
0 10
<re.Match object; span=(0, 12), match='grid(1, 12).'>
0 11
<re.Match object; span=(0, 11), match='grid(1, 2).'>
0 1
<re.Match object; span=(0, 11), match='grid(1, 3).'>
0 2
<re.Match object; span=(0, 11), match='grid(1, 4).'>
0 3
<re.Match object; span=(0, 11), match='grid(1, 5).'>
0 4
<re.Match object; span=(0, 11), match='grid(1, 6).'>
0 5
<re.Match object; span=(0, 11), match='grid(1, 7).'>
0 6
<re.Match object; span=(0, 11), match='grid(1, 8).'>
0 7
<re.Match object; span=(0, 11), match='grid(1, 9).'>
0 8
<re.Match object; span=(0, 12), match='grid(10, 1).'>
9 0
<re.Match object; span=(0, 13), match='grid(10, 10).'>
9 9
<re.Match object; span=(0, 13), match='grid(10, 11).'>
9 10
<re.Match object; span=(0, 13), match='grid(10, 12).'>
9 11
<re.Match object; span=(0, 12), match='grid(10, 2).'>
9 1
<re.Match object

In [22]:
# End timer
end_time_1 = time.time()



In [23]:
# Calculate elapsed time
elapsed_time = (end_time_1 - start_time_1)/60
elapsed_time

38.91553866465886